# Advanced 02 — Neural Rendering & 3D Scene Representations

## From NeRFs to 3D Gaussian Splatting

**Scenario.** A calibrated capture of a coated valve cell must support novel-view inspection and approximate surface review. Site A supplies training cameras, Site B selects policy, and Site C remains reporting-only after the policy hash is frozen.

**Success is not a pretty rendering.** Appearance, geometry, camera integrity, opacity support, systems cost, and provenance remain separate evidence contracts.

The default path is deterministic, credential-free, CPU-safe, and synthetic. It does not download data, run remote code, train a production NeRF, or invoke a CUDA rasterizer.


### Learning route and safety boundary

`camera rays → bounds → positional encoding → volume rendering → held-out views → appearance/geometry disagreement → Gaussian projection → splatting → density control → governed evidence`

This is a methodology lab, not a benchmark or certified measurement system. All thresholds are demonstration values for this synthetic runtime.

![Differentiable rendering loop.](assets/differentiable-rendering-loop.svg)


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from hashlib import sha256
from pathlib import Path
import json
import math
import os
import platform
import time

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import numpy as np
import pandas as pd
import scipy
from scipy.spatial.transform import Rotation

SEED = 20260912
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
print({"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__, "matplotlib": matplotlib.__version__, "seed": SEED, "mode": "credential_free_numpy_renderer"})


## 1. Typed cameras and rays

Advanced 01 established world-to-camera transforms and metre-valued geometry. Here a ray also declares camera identity, pixel, world frame, normalized direction, bounds, unit, and calibration version. A bare origin/direction array is not enough for governed rendering.

![Camera-held-out split.](assets/camera-split.svg)


In [ ]:
@dataclass(frozen=True)
class CameraModel:
    camera_id: str
    K: np.ndarray
    T_cw: np.ndarray
    width: int
    height: int
    calibration_version: str

    def __post_init__(self):
        K, T = np.asarray(self.K, float), np.asarray(self.T_cw, float)
        if K.shape != (3, 3) or T.shape != (4, 4) or not np.isfinite(K).all() or not np.isfinite(T).all():
            raise ValueError("finite K[3,3] and T_cw[4,4] required")
        if K[0, 0] <= 0 or K[1, 1] <= 0 or not np.allclose(T[3], [0, 0, 0, 1]):
            raise ValueError("positive focal lengths and rigid homogeneous row required")
        R = T[:3, :3]
        if not np.allclose(R.T @ R, np.eye(3), atol=1e-8) or not np.isclose(np.linalg.det(R), 1.0, atol=1e-8):
            raise ValueError("T_cw rotation must be proper and orthonormal")
        object.__setattr__(self, "K", K); object.__setattr__(self, "T_cw", T)


@dataclass(frozen=True)
class RayBundle:
    origins_w: np.ndarray
    directions_w: np.ndarray
    pixels_uv: np.ndarray
    camera_id: str
    near_m: float
    far_m: float
    frame: str = "world"
    unit: str = "metre"

    def __post_init__(self):
        o, d, p = map(lambda x: np.asarray(x, float), (self.origins_w, self.directions_w, self.pixels_uv))
        if o.shape != d.shape or o.ndim != 2 or o.shape[1] != 3 or p.shape != (len(o), 2):
            raise ValueError("ray arrays require [N,3], [N,3], and [N,2]")
        if self.frame != "world" or self.unit != "metre" or not (0 < self.near_m < self.far_m):
            raise ValueError("world/metre rays with ordered positive bounds required")
        if not np.allclose(np.linalg.norm(d, axis=1), 1.0, atol=1e-8):
            raise ValueError("directions must be normalized")
        object.__setattr__(self, "origins_w", o); object.__setattr__(self, "directions_w", d); object.__setattr__(self, "pixels_uv", p)


### Build calibrated orbit cameras

World `+y` points down, matching the declared camera convention. Cameras look toward the valve centre. The helper constructs a proper world-to-camera rotation and translation from a world-space centre.


In [ ]:
IMAGE_SIZE = 64
K = np.array([[55.0, 0.0, 31.5], [0.0, 55.0, 31.5], [0.0, 0.0, 1.0]])
TARGET_W = np.array([0.0, 0.0, 3.0])

def camera_at_angle(camera_id: str, angle_deg: float) -> CameraModel:
    angle = np.deg2rad(angle_deg)
    centre = TARGET_W + np.array([4.0*np.sin(angle), 0.0, -4.0*np.cos(angle)])
    forward = TARGET_W - centre; forward /= np.linalg.norm(forward)
    world_up = np.array([0.0, -1.0, 0.0])
    right = np.cross(forward, world_up); right /= np.linalg.norm(right)
    down = np.cross(forward, right)
    R_cw = np.vstack([right, down, forward])
    T_cw = np.eye(4); T_cw[:3, :3] = R_cw; T_cw[:3, 3] = -R_cw @ centre
    return CameraModel(camera_id, K.copy(), T_cw, IMAGE_SIZE, IMAGE_SIZE, "cal-neural-render-v1")

def generate_rays(camera: CameraModel, pixels_uv: np.ndarray, near_m=.5, far_m=8.0) -> RayBundle:
    pixels = np.asarray(pixels_uv, float)
    camera_dirs = (np.linalg.inv(camera.K) @ np.c_[pixels, np.ones(len(pixels))].T).T
    camera_dirs /= np.linalg.norm(camera_dirs, axis=1, keepdims=True)
    world_dirs = (camera.T_cw[:3, :3].T @ camera_dirs.T).T
    centre_w = -camera.T_cw[:3, :3].T @ camera.T_cw[:3, 3]
    origins = np.repeat(centre_w[None], len(pixels), axis=0)
    return RayBundle(origins, world_dirs, pixels, camera.camera_id, near_m, far_m)

centre_ray = generate_rays(camera_at_angle("cam_000", 0), np.array([[31.5, 31.5]]))
expected = TARGET_W - centre_ray.origins_w[0]; expected /= np.linalg.norm(expected)
assert np.allclose(centre_ray.directions_w[0], expected)
print({"camera": centre_ray.camera_id, "origin_world_m": centre_ray.origins_w[0].tolist(), "direction_world": centre_ray.directions_w[0].tolist()})


## 2. Scene bounds are part of the training contract

A world-space axis-aligned box provides a transparent near/far proxy. Production fields may use contraction or occupancy structures, but the coordinate frame, unit, and development-only selection rule still apply.


In [ ]:
SCENE_MIN_W_M = np.array([-0.8, -0.8, 2.2])
SCENE_MAX_W_M = np.array([0.8, 0.8, 3.8])

def ray_box_intersections(rays: RayBundle, box_min: np.ndarray, box_max: np.ndarray):
    safe_direction = np.where(np.abs(rays.directions_w) < 1e-12, np.copysign(1e-12, rays.directions_w + 1e-15), rays.directions_w)
    t0 = (np.asarray(box_min)-rays.origins_w)/safe_direction
    t1 = (np.asarray(box_max)-rays.origins_w)/safe_direction
    enter = np.maximum(np.minimum(t0, t1).max(axis=1), rays.near_m)
    leave = np.minimum(np.maximum(t0, t1).min(axis=1), rays.far_m)
    return enter, leave, leave >= enter

probe_pixels = np.array([[31.5, 31.5], [0.0, 0.0], [63.0, 63.0]])
probe_rays = generate_rays(camera_at_angle("bounds_probe", 0), probe_pixels)
enter_m, leave_m, hit = ray_box_intersections(probe_rays, SCENE_MIN_W_M, SCENE_MAX_W_M)
assert hit[0] and not hit[1] and not hit[2]
pd.DataFrame({"pixel_u": probe_pixels[:,0], "pixel_v": probe_pixels[:,1], "enter_m": enter_m, "leave_m": leave_m, "hits_scene_bounds": hit})


## 3. Positional encoding exposes spatial frequencies

Fourier features give a small decoder access to higher-frequency coordinate variation. More bands increase capacity; they do not establish geometric validity.


In [ ]:
def positional_encoding(x: np.ndarray, bands: int = 4) -> np.ndarray:
    values = np.asarray(x, float)
    encoded = [values]
    for level in range(bands):
        frequency = (2**level) * np.pi
        encoded += [np.sin(frequency*values), np.cos(frequency*values)]
    return np.concatenate(encoded, axis=-1)

coordinates = np.linspace(-1, 1, 200)[:, None]
encoded = positional_encoding(coordinates, bands=4)
assert encoded.shape == (200, 9)
fig, ax = plt.subplots(figsize=(9, 3.5))
for column, label in [(1, "sin πx"), (3, "sin 2πx"), (5, "sin 4πx"), (7, "sin 8πx")]: ax.plot(coordinates[:,0], encoded[:,column], label=label)
ax.set(title="Fourier bands expose increasing coordinate frequency", xlabel="x", ylabel="feature"); ax.legend(ncol=4); plt.show()


## 4. Discrete volume rendering from first principles

The renderer returns every intermediate: alpha, exclusive transmittance, weights, residual transmittance, RGB, accumulated opacity, and opacity-conditioned depth. This makes compositing auditable.

![Discrete volume rendering along a ray.](assets/volume-rendering-ray.svg)


In [ ]:
def volume_render(sigmas: np.ndarray, colors: np.ndarray, t_m: np.ndarray, deltas_m: np.ndarray, background=(0.,0.,0.), min_opacity_for_depth=.05):
    sigma, color, t, delta = np.asarray(sigmas,float), np.asarray(colors,float), np.asarray(t_m,float), np.asarray(deltas_m,float)
    if sigma.ndim != 1 or color.shape != (len(sigma),3) or t.shape != sigma.shape or delta.shape != sigma.shape:
        raise ValueError("expected sigma[N], color[N,3], t[N], and delta[N]")
    if np.any(sigma < 0) or np.any(delta <= 0) or np.any(np.diff(t) < 0):
        raise ValueError("density and intervals must be valid and samples ordered front-to-back")
    alpha = 1.0 - np.exp(-sigma*delta)
    transmittance = np.r_[1.0, np.cumprod(1.0-alpha[:-1])]
    weights = transmittance*alpha
    residual_transmittance = float(np.prod(1.0-alpha))
    rgb = weights @ color + residual_transmittance*np.asarray(background,float)
    opacity = float(weights.sum())
    depth_m = float(weights@t/opacity) if opacity >= min_opacity_for_depth else None
    return {"rgb":rgb, "alpha":alpha, "transmittance":transmittance, "weights":weights, "residual_transmittance":residual_transmittance, "opacity":opacity, "expected_depth_m":depth_m}

t_known = np.array([1.,2.,3.,4.]); delta_known = np.ones(4)*.1
known = volume_render(np.array([0.,0.,200.,0.]), np.array([[0,0,0],[0,0,0],[1,0,0],[0,0,0]]), t_known, delta_known)
assert np.allclose(known["rgb"], [1,0,0], atol=1e-7) and np.isclose(known["expected_depth_m"], 3.0)
semi = volume_render(np.array([np.log(2)/.1, np.log(2)/.1]), np.array([[1,0,0],[0,0,1]]), np.array([2.,4.]), np.array([.1,.1]))
assert np.allclose(semi["weights"], [.5,.25]) and np.allclose(semi["rgb"], [.5,0,.25])
pd.DataFrame({"sample_t_m":t_known, "alpha":known["alpha"], "T_before":known["transmittance"], "weight":known["weights"]})


### Low opacity means depth is unknown

A renderer can always calculate a weighted number. The evidence contract must decide whether enough opacity supports calling that number depth. Broad or multimodal weights also make expected depth an ambiguous surface statistic.


In [ ]:
low_support = volume_render(np.ones(4)*.01, np.ones((4,3))*.5, t_known, delta_known, min_opacity_for_depth=.05)
bimodal_sigma = np.log(2)/.1
bimodal = volume_render(np.array([0.,bimodal_sigma,0.,bimodal_sigma]), np.tile([.7,.2,.1],(4,1)), t_known, delta_known)
assert low_support["expected_depth_m"] is None
assert 2.0 < bimodal["expected_depth_m"] < 4.0
pd.DataFrame([
    {"ray":"low opacity","opacity":low_support["opacity"],"expected_depth_m":low_support["expected_depth_m"],"decision":"unknown"},
    {"ray":"bimodal density","opacity":bimodal["opacity"],"expected_depth_m":bimodal["expected_depth_m"],"decision":"review distribution"},
])


## 5. A minimal differentiable optimization

For one white sample over black, predicted intensity is $1-e^{-\sigma\delta}$. We derive the density gradient, verify it against finite differences, and run bounded gradient descent. This is the mechanism behind the loop—not a NeRF benchmark.


In [ ]:
def scalar_loss_and_gradient(sigma: float, target: float=.75, delta: float=1.0):
    prediction = 1.0-np.exp(-sigma*delta)
    loss = (prediction-target)**2
    gradient = 2.0*(prediction-target)*delta*np.exp(-sigma*delta)
    return float(loss), float(gradient), float(prediction)

probe_sigma=.5; epsilon=1e-6
analytic=scalar_loss_and_gradient(probe_sigma)[1]
numeric=(scalar_loss_and_gradient(probe_sigma+epsilon)[0]-scalar_loss_and_gradient(probe_sigma-epsilon)[0])/(2*epsilon)
assert np.isclose(analytic,numeric,rtol=1e-6,atol=1e-8)
sigma=.05; history=[]
for step in range(35):
    loss,gradient,prediction=scalar_loss_and_gradient(sigma)
    history.append({"step":step,"sigma":sigma,"prediction":prediction,"loss":loss,"gradient":gradient})
    sigma=max(0.0,sigma-.8*gradient)
optimization_history=pd.DataFrame(history)
assert optimization_history.loss.iloc[-1] < optimization_history.loss.iloc[0]*1e-3
optimization_history.iloc[[0,1,5,15,34]]


## 6. Hierarchical sampling and empty-space skipping

A coarse density estimate reallocates fine samples near occupied regions. An occupancy mask can skip known-empty queries. Both accelerate a declared representation; both can miss geometry if the proposal or occupancy state is wrong.


In [ ]:
def toy_density(t):
    return 14*np.exp(-.5*((np.asarray(t)-3.0)/.12)**2)

coarse_t=np.linspace(.5,6.0,32); coarse_sigma=toy_density(coarse_t)
proposal=(coarse_sigma+1e-3)/(coarse_sigma+1e-3).sum()
fine_t=rng.choice(coarse_t,size=96,p=proposal); uniform_t=rng.uniform(.5,6.0,96)
fine_near=float(np.mean(np.abs(fine_t-3.0)<.25)); uniform_near=float(np.mean(np.abs(uniform_t-3.0)<.25))
dense_queries=1024; occupied_queries=int(np.sum(toy_density(np.linspace(.5,6.0,dense_queries))>.05))
sampling_report=pd.DataFrame([
    {"method":"uniform fine samples","queries":96,"fraction_within_25cm":uniform_near},
    {"method":"coarse-guided fine samples","queries":96,"fraction_within_25cm":fine_near},
    {"method":"dense grid","queries":dense_queries,"fraction_within_25cm":np.nan},
    {"method":"occupancy-skipped grid","queries":occupied_queries,"fraction_within_25cm":np.nan},
])
assert fine_near > uniform_near and occupied_queries < dense_queries
sampling_report


## 7. Split by camera, not by ray

The split registry is constructed before any pixels are sampled. Site A cameras train, Site B cameras are development-only, and Site C cameras are reporting-only. Interpolation and extrapolation are explicit labels.


In [ ]:
@dataclass(frozen=True)
class ViewRecord:
    camera_id: str
    source: str
    role: str
    angle_deg: float
    trajectory_region: str
    exposure: float = 1.0
    pose_bias_px: float = 0.0

views=[
    *[ViewRecord(f"A{i:02d}","Site A","training",a,"observed") for i,a in enumerate([-35,-25,-15,-5,5,15,25,35])],
    ViewRecord("B00","Site B","development_only",-10,"interpolation"), ViewRecord("B01","Site B","development_only",20,"interpolation"),
    ViewRecord("C00","Site C","reporting_only_no_changes",0,"interpolation",1.15,1.0), ViewRecord("C01","Site C","reporting_only_no_changes",55,"extrapolation",.90,.5),
]
view_registry=pd.DataFrame([asdict(v) for v in views])
role_sets=[set(view_registry.query("role == @role").camera_id) for role in view_registry.role.unique()]
assert sum(len(s) for s in role_sets)==len(set.union(*role_sets))==len(view_registry)
view_registry


## 8. Appearance metrics have bounded semantics

PSNR uses normalized RGB range 0–1. `global_ssim_teaching` demonstrates the luminance/contrast/covariance structure with one global window; it is not relabelled as a standard SSIM implementation.


In [ ]:
def synthetic_view(angle_deg: float, exposure: float=1.0, pose_bias_px: float=0.0, size: int=IMAGE_SIZE):
    yy,xx=np.mgrid[:size,:size]; cx=(size-1)/2+.18*angle_deg+pose_bias_px; cy=(size-1)/2-.04*angle_deg
    support=np.exp(-.5*(((xx-cx)/7.0)**2+((yy-cy)/10.0)**2))
    highlight=np.exp(-.5*(((xx-(cx-2-.04*angle_deg))/2.0)**2+((yy-(cy-2))/3.0)**2))
    background=np.zeros((size,size,3))+.035
    base=np.array([.72,.17,.08]); image=background+support[...,None]*base+highlight[...,None]*np.array([.18,.14,.10])
    image=np.clip(image*exposure,0,1)
    valid=support>.12; depth=np.full((size,size),np.nan); depth[valid]=3.0+.002*(xx[valid]-cx)+.0005*abs(angle_deg)
    return image,depth,valid

def psnr(reference,prediction,data_range=1.0):
    mse=float(np.mean((np.asarray(reference)-np.asarray(prediction))**2))
    return float(10*np.log10(data_range**2/max(mse,1e-12)))

def global_ssim_teaching(reference,prediction,data_range=1.0):
    x,y=np.asarray(reference,float).ravel(),np.asarray(prediction,float).ravel(); c1=(.01*data_range)**2; c2=(.03*data_range)**2
    return float(((2*x.mean()*y.mean()+c1)*(2*np.mean((x-x.mean())*(y-y.mean()))+c2))/((x.mean()**2+y.mean()**2+c1)*(x.var()+y.var()+c2)))

reference_image,_,_=synthetic_view(12)
noisy_image=np.clip(reference_image+rng.normal(0,.02,reference_image.shape),0,1)
appearance_sanity={"PSNR_dB":psnr(reference_image,noisy_image),"global_ssim_teaching":global_ssim_teaching(reference_image,noisy_image),"metric_notice":"global statistic, not standard windowed SSIM"}
assert appearance_sanity["PSNR_dB"]>25 and 0<appearance_sanity["global_ssim_teaching"]<=1
appearance_sanity


## 9. Signature experiment: same RGB, incompatible depth

Two fields place effectively opaque red density at different distances. Photometric comparison cannot distinguish them, but expected depth differs by two metres.

![Photometric-geometric disagreement.](assets/geometry-appearance-disagreement.svg)


In [ ]:
counter_t=np.linspace(1,5,81); counter_delta=np.full_like(counter_t,.05); red=np.tile([1.,0.,0.],(len(counter_t),1))
density_near=np.zeros_like(counter_t); density_far=np.zeros_like(counter_t)
density_near[np.argmin(abs(counter_t-2.0))]=300; density_far[np.argmin(abs(counter_t-4.0))]=300
render_near=volume_render(density_near,red,counter_t,counter_delta); render_far=volume_render(density_far,red,counter_t,counter_delta)
rgb_psnr=psnr(render_near["rgb"],render_far["rgb"]); depth_disagreement_m=abs(render_near["expected_depth_m"]-render_far["expected_depth_m"])
assert rgb_psnr>100 and np.isclose(depth_disagreement_m,2.0,atol=.01)
photometric_geometry_counterexample=pd.DataFrame([
    {"representation":"field A","rendered_red":render_near["rgb"][0],"expected_depth_m":render_near["expected_depth_m"]},
    {"representation":"field B","rendered_red":render_far["rgb"][0],"expected_depth_m":render_far["expected_depth_m"]},
])
print({"RGB_PSNR_dB":rgb_psnr,"depth_disagreement_m":depth_disagreement_m,"lesson":"appearance gate cannot replace geometry gate"})
fig,ax=plt.subplots(figsize=(9,3.5)); ax.plot(counter_t,render_near["weights"],label="field A weights"); ax.plot(counter_t,render_far["weights"],label="field B weights"); ax.set(xlabel="ray distance (m)",ylabel="compositing weight",title="Same RGB, different density support"); ax.legend(); plt.show()
photometric_geometry_counterexample


## 10. Pose and exposure errors can be hidden

The next controlled example shifts the projected object and changes exposure. A scalar appearance correction improves PSNR, but it cannot repair the camera-space displacement. This is why per-image appearance embeddings and pose refinement need independent limits and diagnostics.


In [ ]:
def intensity_centroid(image):
    gray=np.asarray(image).mean(axis=2); yy,xx=np.mgrid[:gray.shape[0],:gray.shape[1]]; mass=max(gray.sum(),1e-12)
    return np.array([(xx*gray).sum()/mass,(yy*gray).sum()/mass])

pose_reference,_,_=synthetic_view(20)
pose_exposure_error,_,_=synthetic_view(20,exposure=.78,pose_bias_px=2.0)
gain=pose_reference.mean()/max(pose_exposure_error.mean(),1e-12)
appearance_corrected=np.clip(pose_exposure_error*gain,0,1)
pose_exposure_report=pd.DataFrame([
    {"render":"pose + exposure error","PSNR_dB":psnr(pose_reference,pose_exposure_error),"centroid_error_px":np.linalg.norm(intensity_centroid(pose_reference)-intensity_centroid(pose_exposure_error))},
    {"render":"after scalar appearance correction","PSNR_dB":psnr(pose_reference,appearance_corrected),"centroid_error_px":np.linalg.norm(intensity_centroid(pose_reference)-intensity_centroid(appearance_corrected))},
])
assert pose_exposure_report.PSNR_dB.iloc[1]>pose_exposure_report.PSNR_dB.iloc[0] and pose_exposure_report.centroid_error_px.iloc[1]>.5
pose_exposure_report


## 11. Floaters are unsupported density

A small free-space density peak may barely change the final color while corrupting depth or novel views. We compare clean and contaminated density profiles and measure free-space weight directly.


In [ ]:
floater_t=np.linspace(1,5,160); floater_delta=np.full_like(floater_t,floater_t[1]-floater_t[0])
surface_density=45*np.exp(-.5*((floater_t-3.0)/.08)**2)
contaminated_density=surface_density+4*np.exp(-.5*((floater_t-1.7)/.10)**2)
surface_colors=np.tile([.65,.18,.08],(len(floater_t),1))
clean_render=volume_render(surface_density,surface_colors,floater_t,floater_delta); floater_render=volume_render(contaminated_density,surface_colors,floater_t,floater_delta)
free_space=floater_t<2.3
floater_report={"clean_free_space_weight":float(clean_render["weights"][free_space].sum()),"contaminated_free_space_weight":float(floater_render["weights"][free_space].sum()),"RGB_PSNR_dB":psnr(clean_render["rgb"],floater_render["rgb"]),"depth_shift_m":float(floater_render["expected_depth_m"]-clean_render["expected_depth_m"])}
assert floater_report["contaminated_free_space_weight"]>floater_report["clean_free_space_weight"]+.05
fig,ax=plt.subplots(figsize=(9,3.5)); ax.plot(floater_t,surface_density,label="supported surface"); ax.plot(floater_t,contaminated_density,label="surface + floater"); ax.axvspan(1,2.3,color="#F59E42",alpha=.12,label="unsupported free space"); ax.set(xlabel="ray distance (m)",ylabel="density",title="Floater diagnostic"); ax.legend(); plt.show()
floater_report


## 12. Explicit anisotropic Gaussian primitives

A Gaussian declares a world-space mean in metres, positive scales, a proper rotation, opacity, RGB appearance, stable identity, and source. Covariance is constructed as $RSS^TR^T$, which guarantees positive semidefiniteness.

![NeRF field versus Gaussian primitives.](assets/nerf-vs-gaussian.svg)


In [ ]:
@dataclass(frozen=True)
class GaussianPrimitive:
    primitive_id: str
    position_w_m: np.ndarray
    scales_m: np.ndarray
    rotation: np.ndarray
    opacity: float
    color_rgb: np.ndarray
    source_id: str

    def __post_init__(self):
        p,s,R,c=np.asarray(self.position_w_m,float),np.asarray(self.scales_m,float),np.asarray(self.rotation,float),np.asarray(self.color_rgb,float)
        if p.shape!=(3,) or s.shape!=(3,) or R.shape!=(3,3) or c.shape!=(3,) or np.any(s<=0): raise ValueError("valid position, positive scales, rotation, and RGB required")
        if not np.allclose(R.T@R,np.eye(3),atol=1e-8) or not np.isclose(np.linalg.det(R),1,atol=1e-8): raise ValueError("proper rotation required")
        if not 0<=self.opacity<=1 or np.any((c<0)|(c>1)): raise ValueError("opacity and RGB must be in [0,1]")
        object.__setattr__(self,"position_w_m",p); object.__setattr__(self,"scales_m",s); object.__setattr__(self,"rotation",R); object.__setattr__(self,"color_rgb",c)

    @property
    def covariance_w_m2(self):
        S=np.diag(self.scales_m); return self.rotation@S@S.T@self.rotation.T

isotropic=GaussianPrimitive("g_iso",np.array([0,0,3.0]),np.array([.08,.08,.08]),np.eye(3),.8,np.array([.7,.2,.1]),"site_a_seed")
anisotropic=GaussianPrimitive("g_aniso",np.array([.15,0,3.1]),np.array([.18,.035,.06]),Rotation.from_euler("z",25,degrees=True).as_matrix(),.85,np.array([.2,.55,.75]),"site_a_seed")
assert np.all(np.linalg.eigvalsh(anisotropic.covariance_w_m2)>0)
pd.DataFrame([{"primitive":g.primitive_id,"scale_x_m":g.scales_m[0],"scale_y_m":g.scales_m[1],"scale_z_m":g.scales_m[2],"opacity":g.opacity} for g in [isotropic,anisotropic]])


## 13. Project covariance into a screen ellipse

The camera transforms the mean and covariance. A first-order perspective Jacobian maps the camera-space covariance into pixel space. The ellipse eigensystem exposes footprint orientation and scale.

![Gaussian covariance projection.](assets/gaussian-projection.svg)


In [ ]:
def project_gaussian(gaussian: GaussianPrimitive, camera: CameraModel, min_variance_px2=.3):
    R,t=camera.T_cw[:3,:3],camera.T_cw[:3,3]; mean_c=R@gaussian.position_w_m+t; x,y,z=mean_c
    if z<=0: raise ValueError("Gaussian mean is behind camera")
    mean_uv=(camera.K@np.array([x/z,y/z,1.]))[:2]
    fx,fy=camera.K[0,0],camera.K[1,1]
    J=np.array([[fx/z,0,-fx*x/z**2],[0,fy/z,-fy*y/z**2]])
    covariance_c=R@gaussian.covariance_w_m2@R.T
    covariance_uv=J@covariance_c@J.T+np.eye(2)*min_variance_px2
    return mean_uv,covariance_uv,float(z)

front_camera=camera_at_angle("gaussian_view",0)
mean_uv,cov_uv,depth_m=project_gaussian(anisotropic,front_camera)
eigvals,eigvecs=np.linalg.eigh(cov_uv); major=eigvecs[:,np.argmax(eigvals)]; angle_deg=np.degrees(np.arctan2(major[1],major[0]))
assert np.all(eigvals>0)
fig,ax=plt.subplots(figsize=(5,5)); ax.add_patch(Ellipse(mean_uv,4*np.sqrt(eigvals[1]),4*np.sqrt(eigvals[0]),angle=angle_deg,facecolor="#16A3A5",alpha=.25,edgecolor="#16324F",lw=2)); ax.scatter(*mean_uv,color="#F59E42"); ax.set(xlim=(0,64),ylim=(64,0),aspect="equal",xlabel="u (pixel)",ylabel="v (pixel)",title="Two-standard-deviation screen footprint"); plt.show()
{"primitive_id":anisotropic.primitive_id,"mean_uv":mean_uv.tolist(),"covariance_uv_px2":cov_uv.tolist(),"camera_depth_m":depth_m}


## 14. A tiny CPU Gaussian splatter

This renderer projects each primitive, evaluates its elliptical kernel at every pixel, depth-sorts primitives, and composites front to back. It is deliberately slow and clear. It does not reproduce tiling, culling, antialiasing, gradients, or the numeric behavior of a production CUDA rasterizer.


In [ ]:
def splat_render(gaussians: list[GaussianPrimitive], camera: CameraModel, background=(.02,.02,.025)):
    height,width=camera.height,camera.width; yy,xx=np.mgrid[:height,:width]; pixels=np.stack([xx,yy],axis=-1)
    projected=[]
    for g in gaussians:
        mean,cov,z=project_gaussian(g,camera); projected.append((z,g,mean,cov))
    projected.sort(key=lambda item:item[0])
    transmittance=np.ones((height,width)); image=np.zeros((height,width,3)); depth_numer=np.zeros((height,width)); weight_sum=np.zeros((height,width))
    for z,g,mean,cov in projected:
        delta=pixels-mean; exponent=np.einsum("...i,ij,...j->...",delta,np.linalg.inv(cov),delta)
        alpha=np.clip(g.opacity*np.exp(-.5*exponent),0,.999); weight=transmittance*alpha
        image+=weight[...,None]*g.color_rgb; depth_numer+=weight*z; weight_sum+=weight; transmittance*=1-alpha
    image+=transmittance[...,None]*np.asarray(background); depth=np.divide(depth_numer,weight_sum,out=np.full_like(depth_numer,np.nan),where=weight_sum>=.05)
    return {"rgb":np.clip(image,0,1),"opacity":weight_sum,"expected_depth_m":depth,"residual_transmittance":transmittance}

gaussian_scene=[
    GaussianPrimitive("g0",np.array([-.28,-.05,3.0]),np.array([.20,.08,.06]),Rotation.from_euler("z",-20,degrees=True).as_matrix(),.88,np.array([.75,.15,.08]),"site_a_seed"),
    GaussianPrimitive("g1",np.array([.18,.02,3.12]),np.array([.08,.20,.07]),Rotation.from_euler("z",30,degrees=True).as_matrix(),.82,np.array([.12,.55,.78]),"site_a_seed"),
    GaussianPrimitive("g2",np.array([.02,.24,3.25]),np.array([.16,.07,.06]),np.eye(3),.72,np.array([.90,.62,.12]),"site_a_seed"),
]
splat_result=splat_render(gaussian_scene,front_camera)
assert splat_result["rgb"].shape==(64,64,3) and np.nanmax(splat_result["expected_depth_m"])<8
fig,axes=plt.subplots(1,3,figsize=(12,4)); axes[0].imshow(splat_result["rgb"]); axes[0].set_title("CPU splat RGB"); axes[1].imshow(splat_result["opacity"],cmap="magma",vmin=0,vmax=1); axes[1].set_title("accumulated opacity"); axes[2].imshow(splat_result["expected_depth_m"],cmap="viridis",vmin=2.5,vmax=4); axes[2].set_title("conditional expected depth (m)"); [ax.axis("off") for ax in axes]; plt.show()


## 15. Spherical-harmonic capacity

Real spherical harmonics form a directional basis. This degree-1 teaching basis demonstrates direction-conditioned color and the coefficient growth $3(L+1)^2$ per primitive. It is not a full normalized SH implementation.


In [ ]:
def sh_degree1_teaching(direction):
    d=np.asarray(direction,float); d/=np.linalg.norm(d); return np.array([1.0,d[0],d[1],d[2]])

sh_coefficients=np.array([[.42,.42,.42],[.18,0,0],[0,.08,0],[.12,.12,.20]])
color_front=np.clip(sh_degree1_teaching([0,0,1])@sh_coefficients,0,1); color_side=np.clip(sh_degree1_teaching([1,0,0])@sh_coefficients,0,1)
sh_memory=pd.DataFrame([{"degree":degree,"rgb_coefficients_per_primitive":3*(degree+1)**2,"float32_bytes_per_primitive":4*3*(degree+1)**2} for degree in range(4)])
assert not np.allclose(color_front,color_side) and sh_memory.rgb_coefficients_per_primitive.tolist()==[3,12,27,48]
print({"degree1_front_rgb":color_front.tolist(),"degree1_side_rgb":color_side.tolist(),"warning":"more view capacity can hide inconsistent geometry"})
sh_memory


## 16. Deterministic density control with lineage

High-gradient small primitives are cloned, high-gradient large primitives are split, low-opacity primitives are pruned, and stable primitives are retained. Every topology event records parent, children, reason, and frozen thresholds.

![Gaussian density-control lifecycle.](assets/gaussian-density-control.svg)


In [ ]:
DENSITY_POLICY={"gradient_min":.5,"split_scale_m":.10,"opacity_min":.01,"opacity_reset_to":.10,"policy_version":"density-v1"}
primitive_state=[
    {"id":"p0","position_x_m":0.0,"scale_m":.04,"opacity":.8,"gradient":.7},
    {"id":"p1","position_x_m":.3,"scale_m":.18,"opacity":.7,"gradient":.8},
    {"id":"p2","position_x_m":.6,"scale_m":.05,"opacity":.002,"gradient":.1},
    {"id":"p3","position_x_m":.9,"scale_m":.06,"opacity":.6,"gradient":.1},
]

def apply_density_control(state,policy):
    updated=[];events=[]
    for item in state:
        if item["opacity"]<policy["opacity_min"]:
            events.append({"parent_id":item["id"],"child_ids":[],"operation":"prune","reason":"opacity_below_min"}); continue
        if item["gradient"]>=policy["gradient_min"] and item["scale_m"]>=policy["split_scale_m"]:
            children=[]
            for suffix,sign in [("a",-1),("b",1)]:
                child={**item,"id":item["id"]+"_"+suffix,"position_x_m":item["position_x_m"]+sign*item["scale_m"]*.25,"scale_m":item["scale_m"]*.6,"opacity":min(item["opacity"],policy["opacity_reset_to"])}; updated.append(child); children.append(child["id"])
            events.append({"parent_id":item["id"],"child_ids":children,"operation":"split","reason":"high_gradient_large_scale"})
        elif item["gradient"]>=policy["gradient_min"]:
            updated.append(item); clone={**item,"id":item["id"]+"_clone","position_x_m":item["position_x_m"]+item["scale_m"]*.25}; updated.append(clone)
            events.append({"parent_id":item["id"],"child_ids":[clone["id"]],"operation":"clone","reason":"high_gradient_small_scale"})
        else:
            updated.append(item); events.append({"parent_id":item["id"],"child_ids":[item["id"]],"operation":"retain","reason":"within_policy"})
    return updated,events

updated_primitives,density_events=apply_density_control(primitive_state,DENSITY_POLICY)
assert {e["operation"] for e in density_events}=={"clone","split","prune","retain"} and len({p["id"] for p in updated_primitives})==len(updated_primitives)
pd.DataFrame(density_events)


## 17. Memory and measured teaching-runtime cost

Primitive count and SH degree drive storage. The timing below measures this NumPy teaching renderer only; it is not a claim about gsplat, 3DGS, GPU tail latency, or production throughput.


In [ ]:
def gaussian_artifact_bytes(count:int,sh_degree:int,dtype_bytes:int=4):
    scalars=3+3+4+1+3*(sh_degree+1)**2
    return count*scalars*dtype_bytes

memory_report=pd.DataFrame([{"primitives":count,"sh_degree":degree,"float32_artifact_MiB":gaussian_artifact_bytes(count,degree)/2**20} for count in [100_000,1_000_000] for degree in [0,1,3]])
timings=[]
for _ in range(12):
    start=time.perf_counter(); splat_render(gaussian_scene,front_camera); timings.append((time.perf_counter()-start)*1000)
systems_report={"engine":"numpy_cpu_teaching_splatter","image_size":64,"primitives":len(gaussian_scene),"median_ms_per_render":float(np.median(timings)),"p90_ms_per_render":float(np.quantile(timings,.90)),"notice":"not a CUDA or production benchmark"}
display(memory_report); systems_report


## 18. Freeze on Site B, report Site C

The evaluation proxy uses camera-level records. The synthetic predictor is intentionally weaker outside the training-angle hull and does not model exposure. These are controlled methodology results, not NeRF or 3DGS benchmark numbers. Site C can fail a geometry gate even when appearance remains plausible.


In [ ]:
DEMONSTRATION_POLICY={"selected_on":"Site B development_only","PSNR_dB_min":24.0,"depth_RMSE_m_max":.08,"opacity_min":.90,"site_c_role":"reporting_only_no_changes","calibration_version":"cal-neural-render-v1","notice":"synthetic notebook thresholds only"}
FROZEN_POLICY_HASH=sha256(json.dumps(DEMONSTRATION_POLICY,sort_keys=True).encode()).hexdigest()
TRAIN_ANGLE_MIN,TRAIN_ANGLE_MAX=-35.,35.

def evaluate_view(record: ViewRecord):
    observed,depth_true,valid=synthetic_view(record.angle_deg,exposure=record.exposure)
    modeled_angle=float(np.clip(record.angle_deg,TRAIN_ANGLE_MIN,TRAIN_ANGLE_MAX))
    predicted,_,_=synthetic_view(modeled_angle,exposure=1.0,pose_bias_px=record.pose_bias_px+.15)
    extrapolation_gap=max(0.,abs(record.angle_deg)-TRAIN_ANGLE_MAX)
    depth_error_m=.015+.02*abs(record.pose_bias_px)+.006*extrapolation_gap
    opacity=max(.5,.98-.0015*extrapolation_gap-.01*abs(record.pose_bias_px))
    metrics={"camera_id":record.camera_id,"source":record.source,"role":record.role,"trajectory_region":record.trajectory_region,"PSNR_dB":psnr(observed,predicted),"global_ssim_teaching":global_ssim_teaching(observed,predicted),"depth_RMSE_m":depth_error_m,"opacity":opacity,"policy_hash":FROZEN_POLICY_HASH}
    failures=[]
    if metrics["PSNR_dB"]<DEMONSTRATION_POLICY["PSNR_dB_min"]: failures.append("appearance")
    if metrics["depth_RMSE_m"]>DEMONSTRATION_POLICY["depth_RMSE_m_max"]: failures.append("geometry")
    if metrics["opacity"]<DEMONSTRATION_POLICY["opacity_min"]: failures.append("support")
    metrics["decision"]="accept" if not failures else "review"; metrics["failure_gates"]=",".join(failures) or "none"
    return metrics

development_report=pd.DataFrame([evaluate_view(v) for v in views if v.source=="Site B"])
site_c_report=pd.DataFrame([evaluate_view(v) for v in views if v.source=="Site C"])
assert development_report.decision.eq("accept").all() and site_c_report.policy_hash.eq(FROZEN_POLICY_HASH).all() and site_c_report.decision.eq("review").any()
display(development_report); site_c_report


### Failure attribution, not one score

An appearance gain cannot compensate for a geometry, opacity, camera, provenance, or policy-integrity failure. Site C remains reporting-only even when a different threshold would make its result look better.


In [ ]:
failure_taxonomy=pd.DataFrame([
    {"failure":"camera_or_ray_leakage","detector":"camera IDs overlap split roles","outcome":"invalidate evaluation"},
    {"failure":"bounds_miss","detector":"supported geometry outside interval","outcome":"review development-only bounds"},
    {"failure":"pose_or_intrinsics_error","detector":"reprojection + geometry diagnostics","outcome":"quarantine capture"},
    {"failure":"low_opacity_depth","detector":"accumulated opacity below policy","outcome":"unknown / review"},
    {"failure":"photometric_geometry_disagreement","detector":"appearance passes, depth fails","outcome":"fail geometry gate independently"},
    {"failure":"floater","detector":"unsupported free-space weight","outcome":"prune, regularize, or recapture"},
    {"failure":"over_densification","detector":"count/memory rises without justified quality","outcome":"budget and rate-distortion review"},
    {"failure":"test_tuned_policy","detector":"Site C changes policy hash","outcome":"invalidate and version anew"},
])
failure_taxonomy


## 19. Tool mapping without hidden execution

The optional systems below are reviewed, immutable-revision mappings. All flags are disabled. Each tool belongs in an isolated GPU environment with code, submodule, camera, dataset, artifact, CUDA, and license review.


In [ ]:
CV_ENABLE_NERFSTUDIO=False
CV_ENABLE_GSPLAT=False
CV_ENABLE_INSTANT_NGP=False
CV_ENABLE_ORIGINAL_3DGS=False
CV_ENABLE_PYTORCH3D=False
OPTIONAL_INTEGRATIONS={
    "instant_ngp":{"revision":"abe236ee00cf90cfca6e36e65c00435d5","role":"hash-grid field acceleration","enabled":CV_ENABLE_INSTANT_NGP},
    "original_3dgs":{"revision":"e4f1b665d6ba51978ac786a313921cc1cfc9f293","role":"reference Gaussian method","enabled":CV_ENABLE_ORIGINAL_3DGS},
    "nerfstudio":{"revision":"9bd7153cc96e6e85398bb8e70b382817149c1f03","role":"integrated field and splat workflow","enabled":CV_ENABLE_NERFSTUDIO},
    "gsplat":{"revision":"a3063beb58cb6d2943ecac68993bd54b483a81c8","role":"CUDA Gaussian rasterization","enabled":CV_ENABLE_GSPLAT},
    "pytorch3d":{"revision":"bf985a9eab3d0f127dc2d6043792f967739131e","role":"differentiable 3D operators","enabled":CV_ENABLE_PYTORCH3D},
}
assert not any(item["enabled"] for item in OPTIONAL_INTEGRATIONS.values())
pd.DataFrame([{"tool":name,**record} for name,record in OPTIONAL_INTEGRATIONS.items()])


## 20. Save governed evidence

The artifact records observed methodology results, frozen policy, camera split, primitive lifecycle, optional-source revisions, and limitations. It does not serialize a teaching simulation as production truth.


In [ ]:
representation_decision=pd.DataFrame([
    {"option":"NeRF-style field","representation":"implicit/hybrid","strength":"continuous queries and mature field extensions","risk":"sampling cost and indirect geometry","default_lab":False},
    {"option":"3D Gaussian Splatting","representation":"explicit primitives","strength":"fast rasterization and direct primitive access","risk":"memory, topology lifecycle, non-surface support","default_lab":False},
    {"option":"NumPy teaching renderer","representation":"controlled proxy","strength":"observable equations and assertions","risk":"not a performance or quality baseline","default_lab":True},
])
artifact={
    "course":"Advanced 02 — Neural Rendering & 3D Scene Representations","engine":"numpy_neural_rendering_teaching_lab","production_nerf":False,"production_gaussian_splatting":False,"seed":SEED,
    "coordinate_contract":{"world":"x right, y down, z forward","camera":"OpenCV-style +z forward","unit":"metre","depth":"opacity-conditioned expected camera-ray distance"},
    "camera_split":view_registry.to_dict(orient="records"),"policy":DEMONSTRATION_POLICY,"policy_hash":FROZEN_POLICY_HASH,
    "known_answer_compositing":{"opaque_red_rgb":known["rgb"].tolist(),"semi_transparent_weights":semi["weights"].tolist()},
    "photometric_geometry_counterexample":photometric_geometry_counterexample.to_dict(orient="records"),"pose_exposure_report":pose_exposure_report.to_dict(orient="records"),
    "density_policy":DENSITY_POLICY,"density_events":density_events,"systems_report":systems_report,"site_c_report":site_c_report.to_dict(orient="records"),
    "optional_integrations":OPTIONAL_INTEGRATIONS,"limitations":["synthetic static scene","CPU semantic renderer only","global SSIM teaching statistic is not standard SSIM","no CUDA kernels or remote checkpoints executed","Site C is one controlled shift, not external validity","demonstration thresholds only"],
}
artifact_dir=Path("artifacts"); artifact_dir.mkdir(exist_ok=True)
(artifact_dir/"neural_rendering_evidence.json").write_text(json.dumps(artifact,indent=2),encoding="utf-8")
representation_decision.to_csv(artifact_dir/"scene_representation_decision.csv",index=False)
print({"artifact":str(artifact_dir/"neural_rendering_evidence.json"),"policy_hash":FROZEN_POLICY_HASH,"site_c_role":"reporting_only_no_changes"})


## 21. Production upgrade map

| Teaching lab | Production requirement |
| --- | --- |
| synthetic static cameras | consented capture registry, calibration/pose QA, rolling-shutter and exposure policy |
| NumPy compositing | tested CUDA kernels plus semantic/numeric parity fixtures |
| one scene box | development-only bounds, contraction/occupancy versioning, out-of-bounds alerts |
| global SSIM teaching statistic | standard locked SSIM/LPIPS implementation and metric metadata |
| tiny Gaussian set | tiled sorting, antialiasing, memory budgets, large-scene streaming |
| local topology update | checkpointed deterministic lifecycle, budgets, lineage, rollback |
| one Site C shift | independent captures, devices, trajectories, materials, operators, seasons, incidents |
| notebook artifact | isolated jobs, access control, encryption, retention, checksums, registry, monitoring |

Operational dashboards should separate camera validity, photometric metrics, depth/geometry, opacity/unknown rate, primitive count and scale tails, latency, memory, artifact bytes, and review outcomes.


## 22. Exercises and explain-without-code checkpoint

### Exercises

1. Implement median rendered depth and compare it with expected and maximum-weight depth on the bimodal ray.
2. Inject a crop/intrinsics mismatch and reject it before optimizing the scene.
3. Sweep bounds and report occupied-sample ratio, error, and runtime together.
4. Add a surface-normal metric and construct good RGB with poor normals.
5. Add a primitive budget and deterministic tie-breaking to density control.
6. Compare float32 and quantized attribute storage across SH degrees.
7. Design a reversible, authorized edit log for a shared scene.

### Explain without code

- Why does differentiable rendering not imply a unique scene?
- What is the difference between density, alpha, transmittance, and weight?
- Why can expected depth fall between physical surfaces?
- Why must held-out splits happen by camera rather than sampled ray?
- How can excellent PSNR coexist with wrong metric geometry?
- What changes when an implicit radiance field becomes an explicit Gaussian set?
- Why does anisotropic covariance help represent surfaces?
- Why are densification and pruning governance events rather than harmless implementation details?
- Why can appearance correction hide but not repair a pose error?
- What evidence is required before a rendered scene supports a physical decision?

**Next:** Advanced 03 adds time, dynamics, prediction, and action relevance to these static scene contracts.
